<a href="https://colab.research.google.com/github/nagamamo/machine_learning_for_finance/blob/main/6_2_sentiment_analysis_using_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**6-2 大規模言語モデル（LLM）による企業決算レポートの分析**

　株価は企業の決算情報のみならず、市場参加者の「強気」「弱気」「中立」などのその企業の将来に対する見方に大きく左右されます。本例では、上場企業各社が発表する決算短信情報のPDFファイルからテキストを抽出し、AIによる大規模言語モデル（LLM）によるセンチメント分析、すなわちこの決算短信が「1.Positive」「2.Negative」「3.Newtral」のいずれかであるのかを分析します。

　具体的には、本例は2026年3月にソニー（6758）が発表した決算短信情報を使用します。本ソースコードを実行するには下記の2つの準備が必要となります。

(1）Google AI StudioからAPIキー（無料版）の取得

https://aistudio.google.com/api-keys

(2) ソニー（6758）から決算短信ファイルの取得とGoogle Colabへのアップロードhttps://www.sony.com/ja/SonyInfo/IR/library/presen/er/pdf/25q4_sony.pdf

決算短信ファイルのアップロードは、Google Colabの「ファイル」から一番左のアイコンをクリックしてください。

　まず、GoogleのGemini APIを呼び出すためのgoogle-genai、AIに出力するデータ項目や型（文字列、数値等）を定義するpydantic、そしてpdfファイルから文字データをテキストとして抽出するためのパッケージであるpypdfをインストールします。

In [ ]:
pip install google-genai pydantic pypdf

　続いて、6つのライブラリ、環境変数設定用ライブラリのos、APIを呼びたすためのgenai、APIへのリクエスト設定やデータ構造の型をインポートするためのモジュールtypes、構造化データモデルを定義するためのBaseModel、AIへルールを強制するためのLiteral、PDFファイルからテキスト抽出のためのライブラリであるpypdfをインポートします。

In [ ]:
#[1]ライブラリのインポート
import os #環境変数設定用ライブラリ
from google import genai #AI接続ライブラリ
from google.genai import types #AI指示のためのライブラリ
from pydantic import BaseModel #回答データ構造化出力ライブラリ
from typing import Literal #AIへの規則強制ライブラリ
import pypdf #PDFファイル読込テキスト抽出ライブラリ

　次にセル[2]では1行目で予めGoogle AI Studioから用意したAPIキーを設定し、2行目でGen AI APIとの通信に必要なメインクライアントオブジェクトを初期化します。

In [ ]:
#[2]APIキーと通信窓口の設定
os.environ["GEMINI_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"#APIキーの設定
client = genai.Client() #Gemini APIとの通信窓口の作成

　続いてセル[3]では、本例で行うセンチメント分析を定義します。ここではBaseModelが決算短信情報から出力する結論を["1-positive", "2-negative", "3-neutral"] のいずれか3通りであることをLiteralにより強制します。

In [ ]:
#[3]センチメント分析の定義
class SentimentAnalysis(BaseModel):
    sentiment: Literal["1-positive", "2-negative", "3-neutral"] #決算短信のセンチメントを1, 2, 3のいずれかに厳密に制限）

　本センチメント分析では、AIをgemini-3.8-flashを使用することをセル[4]の4行目で定義します。5行目はプロンプトに決算短信テキスト（report text）を渡す指示を行い、6行目では、Geminiの動作や応答のフォーマットを制御する設定します。7-8行目は、JSON形式での出力指示およびJSON出力フォーマットを指定します。9行目はAIモデルが可能な限り(=0.0)決定論的、一貫性があり、予測可能であることを強制するよう設定します。

In [ ]:
#[4]センチメント分析の入出力設定
def analyze_earnings_report_with_gemini(report_text: str) -> str:#決算短信を文字列(str)で受取り文字列(str)で返す定義
    try: #clientによりgenerate_contentを呼び出し結果を変数 response に格納
        response = client.models.generate_content(
            model="gemini-3.8-flash", #使用するAIモデル
            contents=report_text,#プロンプトに決算報告書テキスト（report_text）を渡す指示
            config=types.GenerateContentConfig( #生成設定オブジェクト
                response_mime_type="application/json", #JSON形式での出力指示
                response_schema=SentimentAnalysis, #JSON出力フォーマット指定
                temperature=0.0 #AI出力の「多様性・ランダム性」設定
            ),
        )
        result = SentimentAnalysis.model_validate_json(response.text) #返されたJSON文字列（response.text）の読み込みとresultへの格納
        return result.sentiment #センチメント分析結果の抽出
    except Exception as e:
        return f"エラー (API): {e}" #エラーを変数eにより捕捉


　最後にソニー（6758）の2026年3月決算短信のセンチメント分析を実行します。セル[5]の2行目でGoogle Colabに一時的にアップロードされた決算短信のPDFファイルを読み込みます。7行目では、この決算短信レポートの冒頭3行をテキスト抽出対象としています。ここで5や10を指定することにより、冒頭5頁、10頁をテキスト抽出対象とすることができます。8行目において、抽出テキストを格納するための空文字列textを作成し、そのtextに順次、ページごとに抽出したテキストを加えていきます。このAIによるセンチメント分析の結果は「1.positive」と表示されており、実際、2026年4月1日の終値3,392円からその後緩やかな上昇が続いています。

In [ ]:
#[5]センチメント分析の実行
if __name__ == "__main__":
    pdf_file_path = "sony.pdf" #決算短信ファイルをファイルパスに読み込みパスを指定
    print("1. PDFからテキスト抽出の実行...")
    extracted_text = ""
    try:
        reader = pypdf.PdfReader(pdf_file_path)
        for page in reader.pages[:3]: #テキスト抽出対象のページ数指定
            text = page.extract_text() #抽出テキスト格納のための空文字列の作成
            if text:
                extracted_text += text + "\n" #抽出文字textに改行（\n）を加えextracted_textに逐次追加
    except FileNotFoundError:
        print(f"エラー: PDFファイル '{pdf_file_path}' が見つかりません。")
        #PDFファイルが検出できないエラーの表示
        exit()
    except Exception as e:
        print(f"PDFの読み込み中にエラーが発生しました: {e}") #P読み込みエラーの表示
        exit()

    if extracted_text.strip(): #抽出テキストの分析
        print("2. センチメント分析の実行..")
        judgment = analyze_earnings_report_with_gemini(extracted_text)
        print(f"\n【分析結果】: {judgment}")
    else:
        print("PDFからテキスト抽出不可")

1. PDFからテキスト抽出の実行...
2. センチメント分析の実行..

【分析結果】: 1-positive
